<a href="https://colab.research.google.com/github/HunterAlpha7/Rhine-Line/blob/main/CatBoost%2BRF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Flood Probability Prediction
### Ensemble: Random Forest + CatBoost
Target: `FloodProbability` (continuous 0–1)
- 70% train | 10% validation | 20% test

In [1]:
!pip install -q catboost
!pip install -q shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import pandas as pd

%matplotlib inline
sns.set(style="whitegrid")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_csv("/content/drive/MyDrive/Research/Flood prediction database/flood.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (50000, 21)


,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,Encroachments,...,DrainageSystems,CoastalVulnerability,Landslides,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors,FloodProbability
0,3,8,6,6,4,4,6,2,3,2,...,10,7,4,2,3,4,3,2,6,0.450
1,8,4,5,7,7,9,1,5,5,4,...,9,2,6,2,1,1,9,1,3,0.475
2,3,10,4,1,7,5,4,7,4,9,...,7,4,4,8,6,1,8,3,6,0.515
3,4,4,2,7,3,4,1,4,6,4,...,4,2,6,6,8,8,6,6,10,0.520
4,3,7,5,2,5,8,5,2,7,5,...,7,6,5,3,3,4,4,3,4,0.475


In [5]:
X = df.drop('FloodProbability', axis=1)
y = df['FloodProbability']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=2/3, random_state=42)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 35000 | Val: 5000 | Test: 10000


Feature Engineering

In [6]:
# 1. Total Risk Score
risk_features = [col for col in df.columns if col != 'FloodProbability']
df['TotalRiskScore'] = df[risk_features].sum(axis=1)

# 2. Infrastructure Vulnerability
df['Infrastructure_Vulnerability'] = df[['DamsQuality', 'DrainageSystems', 'DeterioratingInfrastructure', 'RiverManagement']].sum(axis=1)

# 3. Environmental Degradation
df['Environmental_Degradation'] = df[['Deforestation', 'Siltation', 'WetlandLoss', 'Landslides', 'Watersheds']].sum(axis=1)

# 4. Climate / Natural Hazards
df['Climate_Natural_Hazards'] = df[['MonsoonIntensity', 'TopographyDrainage', 'ClimateChange', 'CoastalVulnerability']].sum(axis=1)

# 5. Monsoon vs Drainage Interaction (Higher = Worse Risk for both)
df['Monsoon_Drainage_Interaction'] = df['MonsoonIntensity'] * df['DrainageSystems']

print(f"New Shape after Feature Engineering: {df.shape}")
df.head()


New Shape after Feature Engineering: (50000, 26)


,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,Encroachments,...,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors,FloodProbability,TotalRiskScore,Infrastructure_Vulnerability,Environmental_Degradation,Climate_Natural_Hazards,Monsoon_Drainage_Interaction
0,3,8,6,6,4,4,6,2,3,2,...,4,3,2,6,0.450,90,25,17,22,30
1,8,4,5,7,7,9,1,5,5,4,...,1,9,1,3,0.475,95,16,29,23,72
2,3,10,4,1,7,5,4,7,4,9,...,1,8,3,6,0.515,103,21,28,22,21
3,4,4,2,7,3,4,1,4,6,4,...,8,6,6,10,0.520,104,15,29,14,16
4,3,7,5,2,5,8,5,2,7,5,...,4,4,3,4,0.475,95,20,16,24,21


In [7]:
#Initialize the Imputer and Scaler
imputer = SimpleImputer(strategy='mean')
scaler = StandardScaler()

# 2. Fit and transform the TRAINING data
# (Fitting only on the train set prevents data leakage from the test set)
X_train_imputed = imputer.fit_transform(X_train)
X_train_scaled = scaler.fit_transform(X_train_imputed)

# 3. Transform the VALIDATION and TEST data using the rules learned from the train set
X_val_imputed = imputer.transform(X_val)
X_val_scaled = scaler.transform(X_val_imputed)

X_test_imputed = imputer.transform(X_test)
X_test_scaled = scaler.transform(X_test_imputed)

# 4. Convert arrays back to DataFrames to preserve column names for feature importance plotting
X_train = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_val = pd.DataFrame(X_val_scaled, columns=X.columns, index=X_val.index)
X_test = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

print("Preprocessing complete: Mean Imputation and Standard Scaling applied successfully!")

Preprocessing complete: Mean Imputation and Standard Scaling applied successfully!


In [ ]:
# Random Forest – The stable expert
rf = RandomForestRegressor(
    n_estimators=600,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
    warm_start=False
)

# CatBoost – The genius self-correcting kid
cat = CatBoostRegressor(
    iterations=2500,
    learning_rate=0.025,
    depth=8,
    random_seed=42,
    verbose=200,
    loss_function='RMSE',
    eval_metric='RMSE'
)

# Train both
rf.fit(X_train, y_train)
cat.fit(X_train, y_train)

# Ensemble – just average them
ensemble = VotingRegressor([('rf', rf), ('cat', cat)])
ensemble.fit(X_train, y_train)

print("CatBoost + Random Forest ensemble ready!")

0:	learn: 0.0496061	total: 56.6ms	remaining: 2m 21s
200:	learn: 0.0190391	total: 2.13s	remaining: 24.3s
400:	learn: 0.0095631	total: 4.44s	remaining: 23.2s
600:	learn: 0.0053452	total: 6.18s	remaining: 19.5s
800:	learn: 0.0032840	total: 7.47s	remaining: 15.8s
1000:	learn: 0.0021729	total: 8.74s	remaining: 13.1s
1200:	learn: 0.0015810	total: 9.97s	remaining: 10.8s
1400:	learn: 0.0012769	total: 11.2s	remaining: 8.77s
1600:	learn: 0.0011157	total: 12.4s	remaining: 6.99s
1800:	learn: 0.0010285	total: 13.6s	remaining: 5.29s
2000:	learn: 0.0009736	total: 14.9s	remaining: 3.71s
2200:	learn: 0.0009344	total: 16.8s	remaining: 2.28s
2400:	learn: 0.0009044	total: 18.9s	remaining: 777ms
2499:	learn: 0.0008888	total: 19.8s	remaining: 0us


K-Fold validation

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

print("Running 5-Fold Cross-Validation...")

# Setup a 5-Fold split on the training data
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Calculate R2 scores across all 5 folds using the ensemble
cv_r2_scores = cross_val_score(ensemble, X_train, y_train, cv=kf, scoring='r2', n_jobs=-1)

print(f"Cross-Validation R² Scores: {cv_r2_scores}")
print(f"Average Robust R² Score: {cv_r2_scores.mean():.4f} (± {cv_r2_scores.std():.4f})")

### Performance (Expect R² ≥ 0.874)

In [ ]:
y_pred = ensemble.predict(X_test)

print(f"Test RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):.6f}")
print(f"Test MAE  : {mean_absolute_error(y_test, y_pred):.6f}")
print(f"Test R²   : {r2_score(y_test, y_pred):.6f}")

### Classification View (>0.5 = Flood)

### Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

threshold = 0.5
y_test_bin = (y_test >= threshold).astype(int)
y_pred_bin = (y_pred >= threshold).astype(int)

cm = confusion_matrix(y_test_bin, y_pred_bin)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Flood', 'Flood'], yticklabels=['No Flood', 'Flood'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

threshold = 0.5
y_test_bin = (y_test >= threshold).astype(int)
y_pred_bin = (y_pred >= threshold).astype(int)

print(f"Accuracy: {accuracy_score(y_test_bin, y_pred_bin):.5f}")
print(f"F1-Score: {f1_score(y_test_bin, y_pred_bin):.5f}")
print(classification_report(y_test_bin, y_pred_bin, target_names=['No Flood', 'Flood']))

### Plots

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(y_test, y_pred, alpha=0.5, color='purple')
ax[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
ax[0].set_xlabel('Actual')
ax[0].set_ylabel('Predicted')
ax[0].set_title(f'Test Predictions (R² = {r2_score(y_test, y_pred):.4f})')

residuals = y_test - y_pred
ax[1].scatter(y_pred, residuals, alpha=0.5, color='teal')
ax[1].axhline(0, color='red', linestyle='--')
ax[1].set_xlabel('Predicted')
ax[1].set_ylabel('Residuals')
ax[1].set_title('Residual Plot')

plt.show()

### Feature Importance (from CatBoost)

In [ ]:
importances = cat.get_feature_importance()
feat_names = X.columns
idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[idx], y=feat_names[idx], palette='viridis')
plt.title('CatBoost Feature Importance')
plt.show()

SHAP implementation

In [ ]:
import shap

print("Generating SHAP Explainability Plot...")

# Initialize the SHAP explainer specifically for the CatBoost model
explainer = shap.TreeExplainer(cat)

# Calculate SHAP values using the test dataset
shap_values = explainer.shap_values(X_test)

# Generate the SHAP Summary Plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP Value Analysis: Feature Impact on Flood Probability", fontsize=16)
plt.tight_layout()
plt.show()